In [26]:
import torch 
import lightning as L
import yaml
import sys, os
sys.path.append('../')
sys.path.append('src')
sys.path.append('byol-a')

from importlib import reload
from pathlib import Path 

from byol_a.common import load_yaml_config




In [54]:
from lightning_scripts import lightning_byola_classifier
reload(lightning_byola_classifier)

BYOLAClassifier = lightning_byola_classifier.BYOLAClassifier
config_path = 'byol-a/config.yaml'
config = load_yaml_config(config_path)

config['model'] = {}
config['hparas'] = {}
config['hparas']['task_loss_params'] = {
    "signal/word_int":
      {"loss_type": 'crossentropyloss',
      "weight": 1.0},                       # init loss is ~6.6 
   "noise/labels_int":
      {"loss_type": 'bcewithlogitsloss',
      "weight": 1.0},                      # init loss is ~200 
    "signal/speaker_int":
      {"loss_type": 'crossentropyloss',
      "weight": 1.0}
    }

config['audio_transforms'] = {} 
config['audio_transforms']['low_snr'] = -10
config['audio_transforms']['high_snr'] = 10
config['audio_transforms']['rms_level'] = 60

config['model']['arch_kwargs'] = {}
config['data'] = {}
config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794,    
                                    "signal/speaker_int": 433} 
task_str = f"word_and_speaker_task"
config['hparas']['batch_size'] = 32
config['hparas']['lr'] = 0.01
config['data']['eval_max'] = 3
config['hparas']['optimizer'] = "AdamW"
config['hparas']['epochs'] = 1
config['num_workers'] = 1 
config['with_noise'] = True

config['hparas']['task_loss_params'] = {key:value for key,value in config['hparas']['task_loss_params'].items() if key in config['model']['arch_kwargs']['num_classes'].keys()}

config['data']['target_keys'] = list(config['model']['arch_kwargs']['num_classes'].keys())


config['classifier_layer'] = 'features.10'


byola = BYOLAClassifier(config)
byola._get_layer_output_dim()

AudioNTT2020(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=512, out_features=2048, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_featur

51200

In [46]:
dl = byola.train_dataloader()

In [47]:
batch = next(iter(dl))

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [50]:
batch[0].shape

torch.Size([32, 1, 32000])

In [74]:
x = torch.randn(32, 1, 32000)
byola(x)

torch.Size([32, 1, 64, 201])
torch.Size([32, 64, 16, 50])


{'signal/word_int': tensor([[-0.2307,  0.7320, -0.1745,  ..., -0.5592,  0.2289, -0.4929],
         [-0.2617,  0.6393, -0.2374,  ..., -0.4642,  0.0480, -0.5049],
         [-0.1578,  0.7384, -0.3053,  ..., -0.4629,  0.1780, -0.3506],
         ...,
         [-0.2413,  0.7693, -0.1148,  ..., -0.3848,  0.2442, -0.4245],
         [-0.0980,  0.6346, -0.1624,  ..., -0.4210,  0.2002, -0.4412],
         [-0.3332,  0.6499, -0.2259,  ..., -0.3758,  0.1953, -0.4264]],
        grad_fn=<AddmmBackward0>),
 'signal/speaker_int': tensor([[ 0.3249, -1.5359,  0.2138,  ..., -0.3904,  0.9129, -0.2410],
         [ 0.3395, -1.5631,  0.2482,  ..., -0.2459,  0.7092, -0.1760],
         [ 0.3397, -1.4646,  0.2427,  ..., -0.3109,  0.7115, -0.3259],
         ...,
         [ 0.3200, -1.4969,  0.3486,  ..., -0.3352,  0.7690, -0.1332],
         [ 0.1569, -1.4715,  0.3679,  ..., -0.3459,  0.7774, -0.1916],
         [ 0.2690, -1.5471,  0.3095,  ..., -0.3474,  0.7411, -0.2444]],
        grad_fn=<AddmmBackward0>)}

In [57]:
from lightning_scripts.jsinV3DataLoader_precombined_batched import CleanSpeechInNoiseValDatasetBatched


eval_speech_h5_path = '/mnt/home/jfeather/ceph/data/training_datasets_audio/jsinV3BalancedProcessed/sr_20000/splits/train_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'

test_dataset = CleanSpeechInNoiseValDatasetBatched(speech_h5_path=eval_speech_h5_path,
                                        target_keys=config['data']['target_keys'],
                                        batch_size=config['hparas']['batch_size'],
                                        )

In [58]:
test_batch = next(iter(test_dataset))

In [60]:
test_batch[0].shape

torch.Size([32, 40000])

In [67]:
test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=1,
        num_workers=config['num_workers'],
        shuffle=False,
        collate_fn=byola.predict_collate_fn
    )

test_batch = next(iter(test_dataloader))
print(test_batch[0].shape)
byola(test_batch[0])

torch.Size([32, 1, 32000])
torch.Size([32, 1, 64, 201])
torch.Size([32, 64, 16, 50])


{'signal/word_int': tensor([[-0.6033,  1.4683, -0.0981,  ..., -0.0067, -0.2558, -0.2084],
         [-0.4933,  0.7209, -0.3141,  ...,  0.3271, -0.0578, -0.0577],
         [-0.2923,  0.6873, -0.2667,  ..., -1.1864,  0.1468, -0.2581],
         ...,
         [-0.3686,  0.9109, -0.0860,  ..., -0.8092, -0.1294,  0.3907],
         [-0.6362,  0.3426, -0.3545,  ..., -0.1482, -0.1700, -0.1776],
         [-0.1230,  0.3206,  0.1055,  ..., -0.3226,  0.0939,  0.2315]],
        grad_fn=<AddmmBackward0>),
 'signal/speaker_int': tensor([[ 0.9474, -1.5725,  0.2645,  ..., -0.5419,  1.5331, -0.7087],
         [ 1.1033, -2.0249,  0.2570,  ..., -0.9377,  1.5177, -1.0373],
         [ 0.8261, -1.8498,  0.0288,  ..., -0.8640,  1.4167, -1.0561],
         ...,
         [ 0.5051, -0.8771,  0.1079,  ..., -0.0744,  0.5731, -1.1077],
         [ 0.5327, -1.2148,  0.8947,  ..., -0.9854,  1.7745, -0.4997],
         [ 0.4578, -0.9817,  0.7635,  ..., -2.0499,  1.6162, -0.4812]],
        grad_fn=<AddmmBackward0>)}